<a href="https://colab.research.google.com/github/joaocanaslopes/Assignments_ML/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ML Exercise: Fine-tuning a CNN on MNIST with PyTorch

This notebook addresses the challenge of fine-tuning a Convolutional Neural Network (CNN) for the MNIST dataset using PyTorch. The goal is to train a model, save its weights, and prepare it for deployment on Hugging Face Spaces with a Gradio interface.

### 1. Setup and Imports

First, we'll install PyTorch and torchvision, then import the necessary libraries.

In [1]:
# Install PyTorch and torchvision if not already installed (usually pre-installed in Colab)
# !pip install torch torchvision torchaudio

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

### 2. Data Loading and Preprocessing

We'll load the MNIST dataset and apply transformations suitable for a CNN, such as converting images to tensors and normalizing them.

In [2]:
# Define transformations for the MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(), # Convert images to PyTorch tensors
    transforms.Normalize((0.1307,), (0.3081,)) # Normalize pixel values (mean and std for MNIST)
])

# Download and load the MNIST training and test datasets
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# Define data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Training data size: {len(train_dataset)}")
print(f"Test data size: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 44.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.5MB/s]


Training data size: 60000
Test data size: 10000


### 3. Define the Convolutional Neural Network (CNN) Model

Here, we define a simple CNN architecture suitable for the MNIST dataset. For this task, we will create a custom CNN. If a specific pre-trained model from `torchvision` (like ResNet or VGG) is desired, additional steps would be needed to adapt it for single-channel (grayscale) 28x28 images and 10 output classes.

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1) # 1 input channel (grayscale), 32 output channels
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Output size: 14x14

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 32 input channels, 64 output channels
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Output size: 7x7

        # Fully connected layer
        # Input features: 64 channels * 7 * 7 (from previous pooling layer)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10) # 10 output classes for digits 0-9

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7) # Flatten the tensor for the fully connected layer
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model and move it to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

print(f"Model will be trained on: {device}")

Model will be trained on: cpu


### 4. Training the Model

We will define the loss function, optimizer, and implement the training loop for a few epochs.

In [4]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Number of epochs to train
num_epochs = 5 # As requested, 'a couple of epochs'

# Training loop
print("Starting training...")
for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(data)
        loss = criterion(outputs, target)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training complete!")

Starting training...
Epoch [1/5], Loss: 0.1326
Epoch [2/5], Loss: 0.0432
Epoch [3/5], Loss: 0.0293
Epoch [4/5], Loss: 0.0209
Epoch [5/5], Loss: 0.0167
Training complete!


### 5. Evaluate the Model

Let's evaluate the model's performance on the test set.

In [5]:
model.eval() # Set the model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculations during evaluation
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy of the model on the {total} test images: {accuracy:.2f}%")

Accuracy of the model on the 10000 test images: 98.48%


### 6. Save Model Weights

Finally, we'll save the trained model's state dictionary to a `.pth` file, as requested. This file can then be used to load the model for inference or deployment.

In [6]:
model_save_path = 'mnist_cnn_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

Model weights saved to mnist_cnn_model.pth
